# 🪐 01 — Data Exploration
**Kepler Exoplanet Detection Project**

This notebook is for initial exploration of the NASA Kepler KOI dataset before training.
It covers:
1. Loading and inspecting the raw KOI catalog
2. Class distribution analysis
3. Exploring key orbital features
4. Visualising raw vs phase-folded light curves
5. Checking data quality issues (NaN rates, flat signals)

## 📦 Install Dependencies

In [ ]:
!pip install lightkurve scikit-learn fsspec==2025.3.0 -q
print('✅ Ready')

## 📥 1. Load the KOI Catalog

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

url = (
    'https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI'
    '?table=cumulative'
    '&select=kepid,kepoi_name,koi_disposition,koi_period,koi_time0bk,'
    'koi_duration,koi_depth,koi_prad,koi_teq,koi_steff,koi_slogg'
    '&format=csv'
)

df = pd.read_csv(url, comment='#')
print(f'✅ Loaded {len(df)} KOI entries')
print(f'Columns: {list(df.columns)}')
df.head()

## 📊 2. Class Distribution

In [ ]:
# Count each disposition class
counts = df['koi_disposition'].value_counts()
print('Class Distribution:')
print(counts)
print()
print(f'Confirmed planets  : {counts.get("CONFIRMED", 0)}')
print(f'False positives    : {counts.get("FALSE POSITIVE", 0)}')
print(f'Candidates         : {counts.get("CANDIDATE", 0)}')

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2196F3', '#F44336', '#FF9800']
counts.plot(kind='bar', ax=ax, color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Kepler KOI Class Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Disposition')
ax.set_ylabel('Count')
ax.set_xticklabels(counts.index, rotation=0)
for i, v in enumerate(counts):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved class_distribution.png')

## 🔭 3. Orbital Feature Distributions

In [ ]:
# Compare orbital period, planet radius, transit depth across classes
df_c  = df[df['koi_disposition'] == 'CONFIRMED']
df_fp = df[df['koi_disposition'] == 'FALSE POSITIVE']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Feature Distributions: Confirmed vs False Positive', fontsize=13, fontweight='bold')

features = [
    ('koi_period',   'Orbital Period (days)', (0, 100)),
    ('koi_prad',     'Planet Radius (Earth radii)', (0, 20)),
    ('koi_depth',    'Transit Depth (ppm)', (0, 10000)),
]

for ax, (col, label, xlim) in zip(axes, features):
    ax.hist(df_c[col].dropna(),  bins=50, alpha=0.6, color='#2196F3', label='Confirmed', density=True)
    ax.hist(df_fp[col].dropna(), bins=50, alpha=0.6, color='#F44336', label='False Positive', density=True)
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_xlim(xlim)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved feature_distributions.png')

## 💡 4. Explore a Raw Light Curve

In [ ]:
import lightkurve as lk

# Download a single confirmed planet light curve — Kepler-90g
print('Downloading raw light curve for Kepler-90 (KIC 11442793)...')
search = lk.search_lightcurve('KIC 11442793', mission='Kepler', cadence='long')
lc = search[0].download()

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle('Kepler-90g (KIC 11442793) — Confirmed Planet', fontsize=13, fontweight='bold')

# Raw light curve
axes[0].plot(lc.time.value, lc.flux.value, color='#2196F3', linewidth=0.5, alpha=0.8)
axes[0].set_title('Raw Light Curve')
axes[0].set_xlabel('Time (BKJD days)')
axes[0].set_ylabel('Flux (electrons/s)')
axes[0].grid(True, alpha=0.3)

# Flattened
lc_flat = lc.flatten(window_length=101)
axes[1].plot(lc_flat.time.value, lc_flat.flux.value, color='#4CAF50', linewidth=0.5, alpha=0.8)
axes[1].set_title('Flattened Light Curve (stellar variability removed)')
axes[1].set_xlabel('Time (BKJD days)')
axes[1].set_ylabel('Normalised Flux')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('raw_vs_flat_lightcurve.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved raw_vs_flat_lightcurve.png')

## 🌀 5. Phase-Folded Transit View

In [ ]:
# Phase-fold at the known orbital period to reveal the transit dip
PERIOD = 14.44912
T0     = 2.2

lc_fold = lc_flat.fold(period=PERIOD, epoch_time=T0)
lc_bin  = lc_fold.bin(bins=201)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lc_fold.phase.value, lc_fold.flux.value,
        '.', color='#90CAF9', alpha=0.3, markersize=2, label='Raw folded points')
ax.plot(lc_bin.phase.value, lc_bin.flux.value,
        color='#2196F3', linewidth=2, label='Binned (201 pts)')
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_title('Phase-Folded Light Curve — Kepler-90g transit dip', fontsize=12, fontweight='bold')
ax.set_xlabel('Phase (fraction of orbital period)')
ax.set_ylabel('Normalised Flux')
ax.set_xlim(-0.5, 0.5)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('phase_folded_transit.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved phase_folded_transit.png')
print(f'Transit depth: ~{(1 - lc_bin.flux.value.min()):.4f} (normalised flux units)')

## 🧹 6. Data Quality Check

In [ ]:
# Sample 50 KOIs and check NaN rates to understand data quality
import lightkurve as lk
from tqdm.notebook import tqdm

df_confirmed = df[df['koi_disposition'] == 'CONFIRMED'].sample(25, random_state=42)
df_fp        = df[df['koi_disposition'] == 'FALSE POSITIVE'].sample(25, random_state=42)
sample       = pd.concat([df_confirmed, df_fp])

nan_rates, failed = [], 0

for _, row in tqdm(sample.iterrows(), total=len(sample), desc='Checking quality'):
    try:
        search = lk.search_lightcurve(f'KIC {int(row["kepid"])}', mission='Kepler', cadence='long')
        if len(search) == 0:
            failed += 1
            continue
        lc = search[0].download()
        lc_flat = lc.flatten(window_length=101)
        lc_fold = lc_flat.fold(period=row['koi_period'], epoch_time=row['koi_time0bk'])
        lc_bin  = lc_fold.bin(bins=201)
        flux = lc_bin.flux.value
        nan_rates.append(np.isnan(flux).mean())
    except Exception:
        failed += 1

print(f'\n📊 Quality Report (sample of 50 KOIs)')
print(f'  Download failures  : {failed}')
print(f'  Mean NaN rate      : {np.mean(nan_rates):.1%}')
print(f'  Max NaN rate       : {np.max(nan_rates):.1%}')
print(f'  % with NaN > 10%   : {(np.array(nan_rates) > 0.1).mean():.1%}  ← filtered out in main notebook')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(nan_rates, bins=20, color='#FF9800', edgecolor='white')
ax.axvline(0.1, color='#F44336', linestyle='--', linewidth=2, label='10% filter threshold')
ax.set_title('NaN Rate Distribution Across Light Curves', fontsize=12, fontweight='bold')
ax.set_xlabel('NaN fraction')
ax.set_ylabel('Count')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('nan_rate_distribution.png', bbox_inches='tight', dpi=120)
plt.show()
print('✅ Saved nan_rate_distribution.png')

## ✅ Exploration Summary

| Finding | Detail |
|---|---|
| Total KOIs | ~9,500 entries in NASA catalog |
| Confirmed planets | ~2,300 available for training |
| False positives | ~5,000 available for training |
| Balanced sample used | 200 + 200 = 400 total |
| Typical NaN rate | Low — most light curves are usable |
| Transit dip shape | Clear dip visible in phase-folded view |
| Key challenge | Some KIC IDs unresolvable in MAST |

**→ Proceed to `exoplanet_detection_colab.ipynb` for full training pipeline.**